# Setup and Imports

In [1]:
import pandas as pd
from scipy.stats import kendalltau, wilcoxon
import os
import numpy as np

# This is a good practice for reproducibility and to know where your notebook is looking for files.
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content


# Data Loading and Preparation

In [5]:
def load_centrality_data(filepath):
    """
    Loads centrality data from a text file, skipping the first line.
    The file is expected to have two columns: Node_ID and Score.
    """
    try:
        # We use read_csv, which is versatile enough for space-separated files.
        # We specify the separator as a regular expression for one or more spaces.
        # We skip the first row and name the columns for clarity.
        data = pd.read_csv(filepath, sep='\s+', skiprows=1, header=None, names=['NodeID', 'Score'])
        # It's good practice to set the NodeID as the index for easier merging later.
        data = data.set_index('NodeID')
        return data
    except FileNotFoundError:
        print(f"Error: The file was not found at {filepath}")
        return None
    except Exception as e:
        print(f"An error occurred while reading {filepath}: {e}")
        return None

# --- You need to define the paths to your data here ---
# Create a dictionary to hold the paths to your files for easy access.
data_directory = '/content/' # CHANGE THIS if your folder has a different name

file_pairs = {
    'Degree': ('ebi-intact.ungraph.degree.txt', 'ebi-intact.cmty.degree.txt'),
    'Betweenness': ('ebi-intact.ungraph.betweenness.txt', 'ebi-intact.cmty.betweenness.txt'),
    'Closeness': ('ebi-intact.ungraph.closeness.txt', 'ebi-intact.cmty.closeness.txt'),
    'Farness': ('ebi-intact.ungraph.farness.txt', 'ebi-intact.cmty.farness.txt'),
    'Harmonic': ('ebi-intact.ungraph.harmonic.txt', 'ebi-intact.cmty.harmonic.txt'),
    'PageRank': ('ebi-intact.ungraph.pagerank.txt', 'ebi-intact.cmty.pagerank.txt')
}

# Now, let's load all the data.
centrality_data = {}
for measure, (graph_file, hypergraph_file) in file_pairs.items():
    graph_path = os.path.join(data_directory, graph_file)
    hypergraph_path = os.path.join(data_directory, hypergraph_file)

    graph_df = load_centrality_data(graph_path)
    hypergraph_df = load_centrality_data(hypergraph_path)

    if graph_df is not None and hypergraph_df is not None:
        # Merge the two dataframes on the NodeID. This aligns the scores for each protein.
        # An inner join ensures we only compare proteins present in both calculations.
        merged_df = graph_df.merge(hypergraph_df, left_index=True, right_index=True, suffixes=('_graph', '_hypergraph'))
        centrality_data[measure] = merged_df

# Let's inspect one of the merged dataframes to ensure it looks correct.
if 'Degree' in centrality_data:
    print("Sample of merged Degree Centrality data:")
    print(centrality_data['Degree'].head())

Sample of merged Degree Centrality data:
        Score_graph  Score_hypergraph
NodeID                               
0               4.0               5.0
1               3.0               4.0
2               2.0               3.0
3               3.0               3.0
4              25.0              26.0


# Performing the Statistical Analysis

In [7]:
results = []

for measure, df in centrality_data.items():
    ranked_list_graph = df['Score_graph']
    ranked_list_hypergraph = df['Score_hypergraph']

    # --- Kendall's Tau ---
    tau, tau_p_value = kendalltau(ranked_list_graph, ranked_list_hypergraph)

    # --- Wilcoxon Signed-Rank Test ---
    try:
        wilcox_stat, wilcox_p_value = wilcoxon(ranked_list_graph, ranked_list_hypergraph, zero_method='pratt')
    except ValueError:
        wilcox_stat, wilcox_p_value = (np.nan, np.nan)

    results.append({
        'Centrality Measure': measure,
        "Kendall's Tau": tau,
        "Kendall's p-value": tau_p_value, # It's good practice to save this p-value as well
        "Wilcoxon Statistic": wilcox_stat,
        "Wilcoxon p-value": wilcox_p_value
    })

results_df = pd.DataFrame(results)

print("\n--- Statistical Comparison Results ---")
print(results_df)


--- Statistical Comparison Results ---
  Centrality Measure  Kendall's Tau  Kendall's p-value  Wilcoxon Statistic  \
0             Degree       0.291324      2.816814e-110            273142.5   
1        Betweenness       0.979841       0.000000e+00            804860.5   
2          Closeness       1.000000       0.000000e+00                 0.0   
3            Farness       1.000000       0.000000e+00                 0.0   
4           Harmonic       1.000000       0.000000e+00                 0.0   
5           PageRank       0.380782      7.381981e-241            125804.0   

   Wilcoxon p-value  
0      0.000000e+00  
1      6.275621e-23  
2               NaN  
3               NaN  
4      0.000000e+00  
5      0.000000e+00  


/usr/local/lib/python3.11/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


In [8]:
output_filename = 'centrality_comparison_results.csv'

# Use the to_csv method to save the DataFrame
# index=False prevents pandas from writing the DataFrame index (0, 1, 2, ...) as a column
results_df.to_csv(output_filename, index=False)

print(f"\nResults successfully exported to {output_filename}")


Results successfully exported to centrality_comparison_results.csv
